# Pair-2 offline analysis package

**Descriptive and post-hoc. No gate input and no design authority.**
Run `pilot0-pair2-seed29-20260830T204211Z`, seed 29; selected Stage-A
B-S@40 and B-G@20. Only existing pair-2 records and the 34 local adapter
archives are read. No remote calls, new sampling, or pair-2 inspection/control.

The package contains (1) paired item-bootstrap intervals, (2) F2 failure/length
trajectories, (3) early-window F1 statistics, (4) per-layer and cadence adapter
geometry, (5) baseline item-set overlap, and (6) separately labelled non-binding
confirmatory notes. Rate/score definitions, denominator counts, uncertainty
assumptions, and exact source paths are retained in each section.

Code cells were executed sequentially with the repository Python interpreter,
not through a Jupyter kernel service. Execution counts and stdout are saved.
The adjacent Markdown cells are the report snapshot from that execution.
Rerunning a code cell rebuilds its JSON/Markdown; rerun `build_package.py` to
refresh this notebook's saved Markdown snapshots and the combined report.

Dependencies are the repository's existing NumPy, PyTorch and safetensors;
no package was installed. Geometry execution reads about 12.9 GB of local
adapter files and uses one CPU thread for the selected tensor spot checks.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Find this package whether opened at the repo root or inside the package.
here = Path.cwd().resolve()
repo = next(p for p in (here, *here.parents) if (p / "duraseed_pilot_config.yaml").exists())
package = repo / "artifacts/pilot0-pair2-offline-analysis"
python = repo / ".venv/bin/python"
env = dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")

def execute_local(script):
    result = subprocess.run([str(python), str(package / script)], cwd=repo,
                            env=env, capture_output=True, text=True, check=True)
    if result.stderr:
        print(result.stderr)
    print(f"{script}: completed locally, exit {result.returncode}")

print("Pair-2 artifact paths only; no remote clients or new samples.")


Pair-2 artifact paths only; no remote clients or new samples.


In [2]:
execute_local("uncertainty.py")
data = json.loads((package / "uncertainty.json").read_text())
print(json.dumps({"contrasts": len(data["contrasts"]), "bootstrap": data["bootstrap"], "early_window_rows": len(data["early_window"])}, indent=2))


uncertainty.py: completed locally, exit 0
{
  "contrasts": 14,
  "bootstrap": {
    "method": "paired item-cluster percentile",
    "replicates": 50000,
    "base_seed": 314159,
    "confidence": 0.95,
    "multiple_comparison_adjustment": "none; pointwise descriptive intervals",
    "resampling_unit": "item ID shared across both methods and all time points; all draws stay in each item"
  },
  "early_window_rows": 8
}


# Paired item-level uncertainty and early-window F1

Descriptive, post-hoc, seed 29 only. No result enters a gate or changes the running experiment.

## Paired uncertainty

All contrasts are B-S minus B-G. Intervals are 95% paired item-cluster percentile bootstrap CIs with 50,000 replicates (NumPy PCG64, base seed 314159; seeds 314160 and 314161 for the monitor and pre-B panels). Sampling the same item index in both arms preserves pairing; for AUC, that index retains the entire trajectory and all within-item draws. No individual completions are treated as independent items.

The sampling unit is an item, conditional on these fitted models, their selected checkpoints, the fixed family panel, and the observed per-item draw records. Items are treated as exchangeable independent clusters within each analyzed population. This estimates variability over empirical item clusters with realized draws retained, not fresh-completion uncertainty for the exact fixed panel. This is not a family-cluster or training-seed CI; correlation across items in a family could make intervals too narrow. The monitor has 12 assigned families per role. Intervals do not include training-seed variance, checkpoint-selection uncertainty, uncertainty from choosing these post-hoc windows, or a separate within-item resampling model. They are pointwise, without multiplicity adjustment. A posterior-score bootstrap CI is not a Bayesian credible interval.

| Contrast | Paired items | Estimate | 95% CI lower | 95% CI upper |
|---|---:|---:|---:|---:|
| F2 update-480 Pass@1 | 512 | -0.10009766 | -0.11828613 | -0.08264160 |
| F2 update-480 posterior mean | 512 | -0.09420956 | -0.11132813 | -0.07778033 |
| F2 raw-gain AUC (stored posterior-score estimand) | 512 | -0.05688848 | -0.06862360 | -0.04530086 |
| F2 Pass@1 gain AUC (supplementary raw-rate estimand) | 512 | -0.06044401 | -0.07291257 | -0.04813217 |
| F1 targeted monitor update-1 Pass@1 | 192 | -0.10286458 | -0.14062500 | -0.06510417 |
| F1 targeted monitor update-1 posterior mean | 192 | -0.08229167 | -0.11250000 | -0.05208333 |
| F1 targeted monitor update-2 Pass@1 | 192 | -0.11197917 | -0.13802083 | -0.08723958 |
| F1 targeted monitor update-2 posterior mean | 192 | -0.08958333 | -0.11041667 | -0.06979167 |
| F1 targeted monitor update-5 Pass@1 | 192 | -0.04557292 | -0.06119792 | -0.02994792 |
| F1 targeted monitor update-5 posterior mean | 192 | -0.03645833 | -0.04895833 | -0.02395833 |
| F1 targeted monitor update-10 Pass@1 | 192 | -0.02083333 | -0.03255208 | -0.00911458 |
| F1 targeted monitor update-10 posterior mean | 192 | -0.01666667 | -0.02604167 | -0.00729167 |
| Pre-B 16-draw targeted Pass@1 (matching imperfection) | 256 | -0.02441406 | -0.05322266 | 0.00512695 |
| Pre-B 16-draw targeted posterior mean (matching imperfection) | 256 | -0.02297794 | -0.05009191 | 0.00482537 |

Units are proportions (multiply by 100 for percentage points). F2 raw-gain AUC here is the existing posterior-score estimand: trapezoidal absolute AUC over updates 0–480, normalized by 480, minus that arm's own update-0 posterior score. The supplementary Pass@1-gain AUC applies the same operation to raw rates. With 16 draws, posterior differences and gain differences equal 16/17 times their raw-rate counterparts.

Pre-B uses the separate 256-targeted-item × 16-draw A-validation panel, not the one-draw cadence matching panel (17/96 for each selected checkpoint). Its CI is conditional on B-S@40 and B-G@20 having been selected; it does not certify an equivalence margin or adjust for selection.

## Early-window retention

Relative retention is the ratio of the panel's mean score at t to its own update-0 mean; it is not a mean of itemwise ratios and is not clipped at 1. Both raw Pass@1 and the stored Jeffreys posterior score are shown. Half-life is the first downward crossing of half the initial score, linearly interpolated between the adjacent observed checkpoints on the update axis; it is not an exponential-fit parameter or an observed intermediate checkpoint. All 11 available points (0–480) are searched.

AUC(0–20) uses only the observed grid [0,1,2,5,10,20], trapezoidal integration, divided by 20. Relative AUC additionally divides by that arm/role's initial score; the unnormalized absolute area is in the JSON. These are point estimates, without bootstrap intervals.

| Arm / role / score | Relative u1 | u2 | u5 | u10 | Half-life (updates) | Absolute AUC 0–20 | Relative AUC 0–20 |
|---|---:|---:|---:|---:|---|---:|---:|
| B-S / targeted / Pass@1 | 0.555556 | 0.148148 | 0.029630 | 0.022222 | 1.136364 [1, 2] | 0.01438802 | 0.08185185 |
| B-S / targeted / posterior mean | 0.740260 | 0.502165 | 0.432900 | 0.428571 | 2.093750 [2, 5] | 0.11151042 | 0.46341991 |
| B-G / targeted / Pass@1 | 1.013158 | 0.697368 | 0.256579 | 0.125000 | 3.343284 [2, 5] | 0.04820964 | 0.24358553 |
| B-G / targeted / posterior mean | 1.008065 | 0.814516 | 0.544355 | 0.463710 | 7.750000 [5, 10] | 0.13856771 | 0.53639113 |
| B-S / sentinel / Pass@1 | 0.782609 | 0.369565 | 0.260870 | 0.065217 | 1.684211 [1, 2] | 0.01064453 | 0.17771739 |
| B-S / sentinel / posterior mean | 0.929577 | 0.795775 | 0.760563 | 0.697183 | Not crossed (0–480) | 0.10851563 | 0.73362676 |
| B-G / sentinel / Pass@1 | 1.019737 | 0.802632 | 0.197368 | 0.125000 | 3.500000 [2, 5] | 0.04833984 | 0.24424342 |
| B-G / sentinel / posterior mean | 1.012097 | 0.879032 | 0.508065 | 0.463710 | 5.909091 [5, 10] | 0.13867188 | 0.53679435 |

For B-S sentinel posterior retention, half the initial score lies below the four-draw Jeffreys floor of 0.1, so the requested posterior half-life is unattainable under this score definition; the raw-rate half-life is reported separately. Relative retention does not subtract M0 ability and does not measure survival restricted to newly acquired items.

Sources: [pair-2 result](/Users/elyb/Documents/DuraSeed-v1/runs/pilot0/pilot0-pair2-seed29-20260830T204211Z/result.json); [matching](/Users/elyb/Documents/DuraSeed-v1/runs/pilot0/pilot0-pair2-seed29-20260830T204211Z/seed-29/matching.json); [all exact estimates and evaluation paths](uncertainty.json).

Checks: identical paired item-ID and draw populations at every included checkpoint; existing stored F2 curves/AUC contrast reproduced; raw-to-posterior 16/17 identity checked. No remote calls or new samples.


In [3]:
execute_local("f2_diagnostics.py")
data = json.loads((package / "f2_diagnostics.json").read_text())
print(json.dumps({"checks": data["checks"], "baseline_solved_items": data["baseline_concentration"]["baseline_solved_item_count"]}, indent=2))


f2_diagnostics.py: completed locally, exit 0
{
  "checks": {
    "evaluations": 22,
    "completion_reward_joins": 180224,
    "unique_item_draws_per_evaluation": 8192,
    "shared_manifest_items": 512,
    "manifest_ids_verified": true,
    "generation_reward_values_verified": true
  },
  "baseline_solved_items": 249
}


# Pair-2 F2 failure, length, and baseline concentration diagnostics

Local-only descriptive analysis; no new sampling, no gate or design authority.

## Definitions and provenance

Pass@1 is exact-success count / completion count. Length-stop is `sampled_tokens >= sampling_max_tokens`, matching the archived profile reducer; the recorded stop_reason counts are also retained. Missing-tag is the recorded `missing_answer_tag` failure code; invalid-tag is `!valid_answer_tag`; syntactic invalidity is `!valid_syntax` (MAPS valid_program). These are distinct from all verification failures, which also include legal-but-wrong-target programs. Program-too-long concerns MAPS instruction count, not the 128-token cap. Length uses stored sampled_tokens, not retokenization. Definitions follow `src/duraseed/tasks/maps/verifier.py` and `src/duraseed/pilot0_profiles.py` without changing either.

Manifest: `runs/pilot0/pilot0-pair2-seed29-20260830T204211Z/pilot-inputs/b_validation_manifest.json`. Every checkpoint has 512 manifest items × 16 draws = 8,192 completions, with a 128-token cap. Sample-id/task-id joins, draw uniqueness, item sets, and manifest IDs are checked. Generation token IDs and log-probabilities are not retained by this analysis.

All exact source paths, item IDs, counts, rates, and cross-tabs are in [`f2_diagnostics.json`](f2_diagnostics.json). Rates below are percentages of all 8,192 completions unless stated otherwise.

## B-S: completion trajectories

| Update | Correct | Pass@1 % | Cap % | Missing tag % | Invalid tag % | Invalid syntax % | All failure % | Tokens mean | Median |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0 | 0.0000 | 20.0195 | 40.9668 | 100.0000 | 100.0000 | 100.0000 | 61.3607 | 47.0 |
| 1 | 26 | 0.3174 | 15.8813 | 15.7593 | 20.5078 | 91.4795 | 99.6826 | 42.5171 | 22.0 |
| 2 | 453 | 5.5298 | 0.0488 | 1.3306 | 1.8799 | 7.4341 | 94.4702 | 12.3934 | 13.0 |
| 5 | 569 | 6.9458 | 0.0000 | 0.0000 | 0.0000 | 0.1343 | 93.0542 | 12.8647 | 13.0 |
| 10 | 598 | 7.2998 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.7002 | 12.8850 | 13.0 |
| 20 | 587 | 7.1655 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.8345 | 12.4712 | 13.0 |
| 40 | 634 | 7.7393 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.2607 | 12.6541 | 13.0 |
| 80 | 595 | 7.2632 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.7368 | 12.7351 | 13.0 |
| 160 | 604 | 7.3730 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.6270 | 12.7021 | 13.0 |
| 320 | 1125 | 13.7329 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 86.2671 | 12.7649 | 13.0 |
| 480 | 2200 | 26.8555 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 73.1445 | 12.8713 | 13.0 |

### B-S: failure-code counts

| Update | empty_answer | illegal_instruction | invalid_program | missing_answer_tag | multiple_answer_tags | program_too_long | wrong_target |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 0 | 0 | 4832 | 3356 | 4 | 0 | 0 |
| 1 | 0 | 4 | 3899 | 1291 | 389 | 2227 | 356 |
| 2 | 2 | 147 | 478 | 109 | 15 | 6 | 6982 |
| 5 | 0 | 44 | 11 | 0 | 0 | 0 | 7568 |
| 10 | 0 | 1 | 0 | 0 | 0 | 0 | 7593 |
| 20 | 0 | 0 | 0 | 0 | 0 | 0 | 7605 |
| 40 | 0 | 0 | 0 | 0 | 0 | 0 | 7558 |
| 80 | 0 | 3 | 0 | 0 | 0 | 0 | 7594 |
| 160 | 0 | 0 | 0 | 0 | 0 | 0 | 7588 |
| 320 | 0 | 112 | 0 | 0 | 0 | 0 | 6955 |
| 480 | 0 | 230 | 0 | 0 | 0 | 0 | 5762 |

## B-G: completion trajectories

| Update | Correct | Pass@1 % | Cap % | Missing tag % | Invalid tag % | Invalid syntax % | All failure % | Tokens mean | Median |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | 438 | 5.3467 | 6.9946 | 8.8257 | 13.6597 | 19.3970 | 94.6533 | 24.3339 | 13.0 |
| 1 | 630 | 7.6904 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.3096 | 12.9366 | 13.0 |
| 2 | 409 | 4.9927 | 0.0000 | 0.0000 | 0.0000 | 0.8911 | 95.0073 | 11.7283 | 13.0 |
| 5 | 561 | 6.8481 | 0.0000 | 0.0000 | 0.0000 | 3.2593 | 93.1519 | 12.8270 | 13.0 |
| 10 | 575 | 7.0190 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.9810 | 12.5750 | 13.0 |
| 20 | 614 | 7.4951 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.5049 | 12.6729 | 13.0 |
| 40 | 624 | 7.6172 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 92.3828 | 12.5205 | 13.0 |
| 80 | 1135 | 13.8550 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 86.1450 | 12.8478 | 13.0 |
| 160 | 1404 | 17.1387 | 0.0000 | 0.0977 | 0.0977 | 0.0977 | 82.8613 | 12.6486 | 13.0 |
| 320 | 2708 | 33.0566 | 0.0000 | 0.0000 | 0.0000 | 0.0244 | 66.9434 | 12.7302 | 13.0 |
| 480 | 3020 | 36.8652 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 63.1348 | 12.8655 | 13.0 |

### B-G: failure-code counts

| Update | illegal_instruction | invalid_program | missing_answer_tag | multiple_answer_tags | program_too_long | wrong_target |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | 103 | 531 | 723 | 1 | 846 | 5550 |
| 1 | 3 | 0 | 0 | 0 | 0 | 7559 |
| 2 | 2 | 73 | 0 | 0 | 0 | 7708 |
| 5 | 6 | 267 | 0 | 0 | 0 | 7358 |
| 10 | 4 | 0 | 0 | 0 | 0 | 7613 |
| 20 | 0 | 0 | 0 | 0 | 0 | 7578 |
| 40 | 5 | 0 | 0 | 0 | 0 | 7563 |
| 80 | 1197 | 0 | 0 | 0 | 0 | 5860 |
| 160 | 917 | 0 | 8 | 0 | 1 | 5862 |
| 320 | 118 | 2 | 0 | 0 | 0 | 5364 |
| 480 | 294 | 0 | 0 | 0 | 0 | 4878 |

## Question on record

Is B-G's flat phase (updates 1–40) dominated by cap/format failures that then resolve at the 40–80 takeoff?

Cap/format failure union = completion fails exact verification and is at the token cap, has an invalid answer tag, or has invalid syntax. Categories overlap and this union is not a causal classification.

| B-G update | Failures | Cap/format union | Share of failures % | Wrong target | Cap count | Invalid tag | Invalid syntax |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 7562 | 0 | 0.0000 | 7559 | 0 | 0 | 0 |
| 2 | 7783 | 73 | 0.9379 | 7708 | 0 | 0 | 73 |
| 5 | 7631 | 267 | 3.4989 | 7358 | 0 | 0 | 267 |
| 10 | 7617 | 0 | 0.0000 | 7613 | 0 | 0 | 0 |
| 20 | 7578 | 0 | 0.0000 | 7578 | 0 | 0 | 0 |
| 40 | 7568 | 0 | 0.0000 | 7563 | 0 | 0 | 0 |
| 80 | 7057 | 0 | 0.0000 | 5860 | 0 | 0 | 0 |

Across updates 1–40, cap/format failures account for 0.0000%–3.4989% of failures. At updates 40 → 80, exact successes are 624 → 1135; cap counts 0 → 0; invalid-tag counts 0 → 0; invalid-syntax counts 0 → 0; wrong-target counts 7563 → 5860. These are aggregate trajectories; they do not identify a causal mechanism.

## B-G update-0 solved-item concentration

Solved item = at least one exact success among its 16 stored draws.

B-G update 0: 249/512 items with ≥1 success; 438/8,192 successful completions.

| Arm | Update | Solved items | Overlap | Baseline recall % | Jaccard | Successes on baseline / total | Success share % |
| --- | --- | --- | --- | --- | --- | --- | --- |
| B-S | 80 | 321 | 192 | 77.1084 | 0.507937 | 388 / 595 | 65.2101 |
| B-S | 480 | 357 | 215 | 86.3454 | 0.549872 | 1386 / 2200 | 63.0000 |
| B-G | 80 | 351 | 211 | 84.7390 | 0.542416 | 735 / 1135 | 64.7577 |
| B-G | 480 | 409 | 227 | 91.1647 | 0.526682 | 1875 / 3020 | 62.0861 |

Baseline membership is defined by observed successes in only 16 draws, not a latent ability label. The item sets below refer to these same stored 512 items, without resampling or additional evaluation.

### Complete B-G update-0 solved-item list

| Manifest item index | Task ID | Successes / 16 |
| --- | --- | --- |
| 176 | `sha256:0059c2d481df207846a0613ed2a856d25749153cc5e8b91be0cb83b64bcc4243` | 1 |
| 463 | `sha256:006ac7a194fad4def8cd039248fef73c195feba8e0abc5f448db38372547032d` | 1 |
| 425 | `sha256:00e5e80b39f566fd41e4b9ae2f42e1ddaa4aa6ed52c4af780174db4a07da1de6` | 3 |
| 274 | `sha256:00fdb44402e67ca2b90ff65b6ddb2ef5721b126d52a36ad9d664e02f9ca242ab` | 1 |
| 394 | `sha256:012804d285d58ee9d442d2ab07f3dfd1d8ef1d7bb393ff749b1ac3266518370c` | 1 |
| 262 | `sha256:0399a492a7bc2932b8f336d3bb4cb9520f40c62bf3ae1f343f28e2605087a5eb` | 1 |
| 29 | `sha256:04cf85e19f4f052eed5931b0eba7200abc9bb2606b2cdef8a621c56396b7a4d9` | 1 |
| 343 | `sha256:06095793668dc756906c5035378c77be2a308745d5ce376f531ebcd732490fab` | 2 |
| 128 | `sha256:07a7bf2903462b4f92e8f61c6dd933498dd303729e5584be521286ca97c0334d` | 3 |
| 126 | `sha256:08e3c1a9aa958cf8596109ab3199444fb228aab7f90aec1c932b462788d5ff5d` | 1 |
| 403 | `sha256:09044dc6d7220e657a8601180dd90a12f9eabc6022c101a76b0b991ef29cfeb3` | 3 |
| 11 | `sha256:0a313298794f064db0f43bba5dc0f733a321eed64122cc3a33a252bedafb7bec` | 2 |
| 233 | `sha256:0a3c53f66d0c6e751d54e557f362c92d11bb3ec29bf770700a86cac546f17683` | 1 |
| 106 | `sha256:0acc3a8ee121b2b42fc257cd4d16de7eac4b588c9deb6891ebf776793d251f55` | 1 |
| 292 | `sha256:0b2d3e6f62f63b0b20720e0436b4b999b40664783b0a5f28ce8f5b566e463440` | 1 |
| 258 | `sha256:0b3f24ddad9ffe02afa7b0fded070c664a87c07380a739b129c2c5e689aea43d` | 1 |
| 290 | `sha256:0c40f6f960b633ee6636d907ac9f95a1d69f141ff8b56084a292fa6f0ee50713` | 1 |
| 344 | `sha256:0c8100b260a7f8be76278d066ebdf164c5e286969a832ad34179cba12b6977c8` | 1 |
| 62 | `sha256:0e0c401e2c90fbacbb910fa1adf936b71f806a2d4596e2d2dcd954ec9e6d04b9` | 2 |
| 377 | `sha256:0e727efa6af22da824c8904baf34448d82c2e36a9c3d91c7d90bef7f7f798a9f` | 2 |
| 158 | `sha256:109311ab66af062d3ddf5b4c6c25c4eb1a89488a5fa051582e862f5bfb395a94` | 4 |
| 197 | `sha256:10cfb215fca0f855c903a70177654c7d81bb55322fc7196155f674f30013f3cd` | 2 |
| 7 | `sha256:11608c24a6b4d936cbeb13790be3943c91875fb94eaac4deaf7b067db0df2885` | 3 |
| 146 | `sha256:11ed366c4c47d221806b1b7bba62a464cb66505ad8fb70b419d9f0dcd0319414` | 1 |
| 138 | `sha256:1235eb17efa0ee2128e0ebe50f953b86033982b78998daec9f411fd4de33ed2d` | 1 |
| 220 | `sha256:1244a7a1d861526d03a1f6dff3e3d54e0a7475c7567abe28a54fc71c49842b03` | 1 |
| 266 | `sha256:12479615c0315935a294608dee4cd46ef0c7632d04bcdb1bb05f6fe159d2546c` | 1 |
| 390 | `sha256:12b420aa46ee874ce2a2dd94d87f8f58c0486b9be2df3d2d0585c8c67f2949bf` | 1 |
| 312 | `sha256:13fc9c0826b591ef293f9202eb0feffa602e4120aa8a8605440d6b0fcf66f09d` | 1 |
| 354 | `sha256:13fd22cda8ca12262222db65cca458a09ef8f658ee225a1cecf327440806f98e` | 1 |
| 477 | `sha256:1497a98a82cf82623db9b378888ab2dde2128fa8ffef14ee0ebc21b42cd2d7d8` | 1 |
| 475 | `sha256:15b62811bdea2d18f80505506b7502c39d4c3e1f3fb9796365f4fe6fb758d7e8` | 1 |
| 248 | `sha256:15ccc6971c3dd73a49c708789e31b3bcbe6cfd60c867e5899dfbebe3ef50f01a` | 1 |
| 257 | `sha256:19232b73e1018def6147116a1559106b21d571c1e186145ff3b35cb09aa1353d` | 1 |
| 116 | `sha256:19e87ffa6fe8291fcb62c324dac878934ddac25a7b281fd798e21da5327d0ab2` | 2 |
| 234 | `sha256:1b203b93e524f9939f9110c3b68a38f711f4a7a8eb3bfd093ca60898e5390dd7` | 1 |
| 332 | `sha256:1ba7d2f24d4a3207501ac7730cb21aa43c346cb9e82b3c3b364151b2a5fc6e43` | 3 |
| 381 | `sha256:1bc155050ef28df8125e24ccf3e7ed09a8b576551196afac596e5b474b2b8c68` | 3 |
| 533 | `sha256:1bc334a5ccff204efe1a486c0bd5cdbc4d058fd97aeb556fabef5a98450d4e4f` | 1 |
| 408 | `sha256:1c36f24dbc0228fb922baced82178c8ec7e03c9878f4773fa868e3f2988c7f1a` | 1 |
| 169 | `sha256:1c80e4458c9ee7d072c21c46ff9b3caed11f38704d28df35c5011d0fab028a10` | 4 |
| 466 | `sha256:1d443af90d9751a6468053214857e97cfcc199990703a7cbe3092cf687a11ffe` | 2 |
| 189 | `sha256:1d493678ea6456bc3b35e41fd35ecaf4f41b0bb19c6d2abdcf83da4c26f0d712` | 3 |
| 387 | `sha256:1ee9d7308ab9f8bb0bb74812f4a8c6a4ec48f43702fbc14cf3bedee1db11586a` | 2 |
| 492 | `sha256:1fce5251183465c2c225eb16fdb48c177ad0bbba2fc7580433701f74279dae38` | 1 |
| 537 | `sha256:21a818e46ab5230f1b257c1e5259d488eaac4990a27d6b8df550c8f6d5012b12` | 1 |
| 368 | `sha256:220463e774f28034813efa3ad815f1a66dcd8300b333c06920d521f7355f4228` | 1 |
| 101 | `sha256:239f451ca985993ec54a68a7cb741707b1a556e520e46e897a2b67cde7fb9a3c` | 1 |
| 452 | `sha256:248677c2536133871b41d90cdcf8194e0acce40f0f674087388b8a3a96d031a0` | 2 |
| 243 | `sha256:2551425ce20f0d9115d57fc3b2b650d7c612960b3fb00b089cc1d9ff1f2a7e0c` | 3 |
| 52 | `sha256:28571a9c29f79d19937ee3effde2880ec84339d866502ff3d0ba52dcb6650e62` | 1 |
| 6 | `sha256:2864597813be422c34ab44c292be5d159364d0edb021a1e80af2410682dbac42` | 1 |
| 353 | `sha256:2920c0a9fe7d632f29f7d94d4eff2c3a6f71c06951791d3378cbaa26935ed423` | 1 |
| 385 | `sha256:2cbc55e5d3bcce72ff66cf795a6a6243ecb27515c9ebe2315a2c10bdbac9c52d` | 1 |
| 67 | `sha256:2db9dc334552a9353d35c9dfa45552958a5f703d7f9da98559f3a6a83ecf348e` | 1 |
| 500 | `sha256:2f0da46c520b6503c8632162fc9e05199e21ed007c82d93048004fa7f27a4a60` | 3 |
| 317 | `sha256:3124b5be2d34bdaed1f7a2514ddcff76fb637b62de70e4ce26189d8654755965` | 4 |
| 246 | `sha256:31d1348d6283be76e301033145f59c01783eb6e0c0705039a4e313432c8647ac` | 1 |
| 119 | `sha256:327c39358668074e6286c1e2ff4d6730a63aeb47b825fa26c2b3f2590688b6a3` | 2 |
| 506 | `sha256:380557cc954123a798774a8c13ac00b97edbc26be5fe51887547008e20732e54` | 1 |
| 170 | `sha256:381b7d90cb51a00d1e5b33f3ff509b988f89e95565be3bd0e974b45f9d3a358d` | 1 |
| 388 | `sha256:38acbbbefdef9730dabf843b71ae31adff4a1d17882286e3a09e39149c1e7143` | 2 |
| 482 | `sha256:3b2985c677c5a3251b64fe80dca853abe34aaf95f7e01e9ece41bcfb49e25515` | 1 |
| 451 | `sha256:3bfc82b0e051580e6c605d872ad9ca5965c76e819eca66e588c21be0c13666ee` | 2 |
| 484 | `sha256:3df92d2e1371f4549c6e0967b0d87613b50616ff3961062f4ecac9c4a430c742` | 2 |
| 334 | `sha256:40e0653bf96a1bd0ff9463d135205466dbfaf9b94276f995a29a0de945c7340d` | 3 |
| 291 | `sha256:40f9fd4f9815da1e75ebde422199b0cadaea0298aed54b56adcf9bf276cb4e00` | 6 |
| 497 | `sha256:4105222966f15d4c8ba9094a3e2b12efab5c3345352ad506641282308368d51e` | 2 |
| 426 | `sha256:419b1c15d39c453c15d3009b86aafa478928fbeac937f57223ef0fafd96ebedc` | 2 |
| 473 | `sha256:41cc3555805deac5fe59c8b885e35f9e177c985e24dcbdec79f2f8b9244dfba8` | 2 |
| 124 | `sha256:44cf06bcf675267da910a9ff9c757c8b31005022e66ddf13875d1e61384e454b` | 4 |
| 439 | `sha256:44d6cdaffe83f3b2e27aa81df74c658541be9148ee5b21b433670b1bc7babc48` | 1 |
| 58 | `sha256:45931920c374c5ce102740dbe9b578322e76368ca0fa48dba564dca5c04faf32` | 1 |
| 461 | `sha256:4604c5c195eb2e0bc8c974ab2d9ca8c70154227bb844587c16b69e1c81682b92` | 1 |
| 25 | `sha256:47318ae41e6a23f34efda81388372c3920da9a3d9333947bade667bf1b231a85` | 1 |
| 370 | `sha256:4be66c6d87b2851b897e251fd590e02c7e3a30574dbdc83f9fe65914f56b7ad7` | 1 |
| 490 | `sha256:4cfa2d2ae9b105c6d5c7b5eceb2f3b7f428514a7c1825a38f5c130289fab227e` | 1 |
| 44 | `sha256:4f7459debcf859fbcd6319762577de276920846f89312cb07c2469df220d4611` | 5 |
| 429 | `sha256:4fa7137350e2bd78381abf0160f372749fa9675129599451a971366c19bc5368` | 3 |
| 400 | `sha256:50b9a524aa166ee1484bababc1d826573fcc4dcf20fa1993d0c79f101697d020` | 2 |
| 249 | `sha256:518e004677a008e95631d89d267173f3fd12aedad9420c2438ff4b721e1f91f3` | 3 |
| 526 | `sha256:51ea2d493c9210f7821444558897fa0b6602d5850b8eeccb49132f08427922d9` | 3 |
| 279 | `sha256:527e42b585586a14b6220c790bb72da7e19e23f0d3671d96318bc4afa17ce53d` | 2 |
| 77 | `sha256:52f3082d716c035803eb65bbc467cfa8538fc2dc71433ae88b3b9a37386c7ace` | 1 |
| 333 | `sha256:56b020b33d672093c1fd2f2ea4eab9287440b6819f1d8fe8562c3205f4a40a2b` | 1 |
| 311 | `sha256:575bfec93ed183c68dcf5dc6c4338f55bd9f071750f22c27a9c8420b074802c6` | 1 |
| 536 | `sha256:5849b5405148651d0eccac1e0813b3c6a0c1c93ed19d8ed522e7a8a3c7b00b07` | 2 |
| 53 | `sha256:58c50c5ca87d7b29f218e59902a10d0bfced1f2df26ab8decc882227d08b73b5` | 1 |
| 15 | `sha256:59878078b6a805877eb7bb695d705097896ba4e8c24e64ac35fe78119f7af899` | 1 |
| 499 | `sha256:5b051640b9eb310726a6469f74121764b960def2497521586f428dfcf5b508a0` | 2 |
| 206 | `sha256:5c0b6990f556dc79e25f1e62a1ebb1e54008e8141b09b95303a63b72cd9117cc` | 1 |
| 141 | `sha256:5d0578a4bba4050051cac71e0557549b2ba1b6583616da509b65ff2eac77f81c` | 1 |
| 432 | `sha256:61ec50e58234be5b5cd0a29f0b35dccf1069da488f2b85fc6e10bc2fe38926be` | 1 |
| 348 | `sha256:625d6ab634113924a76413bc139312bb572f0c55d396c40a534ba0bde2a5bcce` | 1 |
| 102 | `sha256:62a6052247e101b5d27e0a002b731eea6e556154131fe83102cac64670a173e4` | 1 |
| 134 | `sha256:65bb9be06e10a636b4d05052f375ebb41fa0f23aa4dca6387307b7f6c38eb076` | 3 |
| 307 | `sha256:6625d0df9baae801c1717aa86c697d52f3a9e74ec9e962790f4202da652b7a76` | 1 |
| 61 | `sha256:6627f43c4d5e3d7b02c741625f1f436153119447c12347f41ef363e7ea20e06b` | 1 |
| 103 | `sha256:66edb66edcf432698181413fd04693b2edcac39beb5b1caa83e32956cbb8e5c6` | 2 |
| 365 | `sha256:67c46b3007947e45946a54dafc48fc3869c26b72e0ea4ab5a3939f828e834c31` | 2 |
| 309 | `sha256:67cafb8f865aeaac3372b65f0725529774ee860b2fdcd822d119a789aeec8c96` | 1 |
| 186 | `sha256:69ecc9935835dd29a277e713b966f53497d01e4113170765b1d70350b4ea3275` | 5 |
| 462 | `sha256:6a5e84c2c5cdeaa17e6833c02afea0797307e71ebf12b201edbe25646a6a2b1c` | 3 |
| 313 | `sha256:6df549c5a41b151ceae0d874ef2fc6f564ca79a9f9ce3d383ae9b8d99b5596b8` | 1 |
| 349 | `sha256:6e5fd3dd0b9a5b10c2cb40deec4ce17ff7dc90e33067ba8f5f78f2a9aceebece` | 2 |
| 304 | `sha256:6fa62188b8972754b30e7337fe4192f10a71decdccc5e13bc73e94f34bb440a0` | 7 |
| 520 | `sha256:70877a4e0e1ec59725fa3663390a529cbf736c87591a42bb4469e08866023832` | 1 |
| 155 | `sha256:715aa48d30589f918b358f2ed6496632a318075cfff55f340a4cd40431e257c7` | 1 |
| 420 | `sha256:7245e627469140a4ead2391b2b723b57e92a15b2375fb576661d095a0726ab07` | 1 |
| 254 | `sha256:741c87db68d0b49ab7c86af874da5bbb1e160fa7363f67a7b4cfd87eb358795e` | 2 |
| 288 | `sha256:74a9dc6ddfd04dca36109143b9c753e10f034b47c8f7a9ceb7c6ef68c09f573f` | 2 |
| 259 | `sha256:74dea212da1d142c27bc5e1ab4ae37203b08f0ec25919128943d164f02b1e9d7` | 2 |
| 481 | `sha256:74ee77321ecd0a138b4968ca490e9fb434abefda8c46f2ca833f1de761954771` | 3 |
| 405 | `sha256:758250530649af07e98c5698e6c51c5a73e1c2a5dc4cf5f198a659923a90962f` | 3 |
| 523 | `sha256:766855283bff10a76b4dea23e0a40cf15b36120e11a53704b946c4fd71f6c5a7` | 2 |
| 527 | `sha256:7775557fd604121d06759bf6057b285b5e505a5e4fd0633d7706fb597a4c730a` | 1 |
| 150 | `sha256:77d7c6ea46fe6e39142cbff547dad2de480acb12225fe8f6b7566bbdff0046ee` | 1 |
| 36 | `sha256:7a4c07ce6fe1f5c80bcdee6d4d832970492f1780d22fd11aa8085979e3dc4256` | 1 |
| 69 | `sha256:7b6fff92eea2dd3a13f68dd62cd5734bbfaf04fdd85e5a4919e2705c2fb8b36e` | 4 |
| 521 | `sha256:7efb43bf3c8e7f6bc43533103bf24a06c8f1a0e2e3a3be7334f90c2de2f5faf2` | 1 |
| 97 | `sha256:8128768f37ffceeafb5e70dd21443381c438553aeea4ac7d49a51aad68ad1393` | 1 |
| 228 | `sha256:8269b405b45e1e4fc52dbaf5b03f07a96cd270a2bb7c4e7e12b4334a0bfc833c` | 4 |
| 147 | `sha256:8486556efc32e2be39f3a0f70e519e670c6e96d5060f92442a6b556e365ed129` | 1 |
| 517 | `sha256:85735b55a2d1692c2f33012c4a403e559bfcd18f7f835dd5483e2de9fcbe2b17` | 2 |
| 360 | `sha256:859d8e748ecf63d4b2e916f45f20fb189b192ccf50625b16a86ae2cf9889fc73` | 2 |
| 240 | `sha256:87be75c15649d86d45a876f9e9ceda04475298bb22a1918ebe9179a6464b215d` | 1 |
| 275 | `sha256:87d3297afd3f514aaae89cbc296d363a9167e5f6c42a1a79804116cb65f2140e` | 1 |
| 301 | `sha256:8912f8bf61c6ec71245c9c53187c7f64cc84230dce5b9ddf09ae77903e060a60` | 2 |
| 252 | `sha256:892814efbbc444c74f5be544d6463047477fd5e263e68d5c0fcbdf959b5fb1d7` | 1 |
| 314 | `sha256:89e3306e6082301f076d6a5f25ff9dce6dea92815892f05b8fe0b45535de6ed1` | 1 |
| 108 | `sha256:8adedb7115554187824069c8521b19dfcda3afa0bb2846979e816f27a09d73b4` | 1 |
| 64 | `sha256:8b0f60b91998e62c6ffaa362ad53a1171a398bcf645e9841cd37ad1784c6c96c` | 2 |
| 398 | `sha256:8c5270735ad98f7f0ef6b0faac2f0ba8b3f7c2cb5cbce42706c474e90e23f0f2` | 1 |
| 192 | `sha256:8c7e68df2f1597224ab460cd56157fee9f05a726ff191f96648be0bcd470c7b6` | 1 |
| 305 | `sha256:8cd8c1325c15f5d805ad270b40661a4c0715b11faa6a996f8296f72f7c485c8a` | 1 |
| 285 | `sha256:8ce10c812308d6828de1815fec512a42b9754c47228ee55bae0e2c787d04cb83` | 1 |
| 232 | `sha256:8ea3212a07687b31c3c795077cd830ed86cf7c7ac24583910b87aa14662de3b5` | 3 |
| 129 | `sha256:8fdf1a48186ea0e8c1346f1d2cb51fccc66c231e93f0593206e7959b3f2fe643` | 1 |
| 373 | `sha256:9021c50b08367cc6f402469ef16fc4bfcada1e694dbf51f6756e128daba0a2c9` | 1 |
| 9 | `sha256:9031e0588bffee9b6c70e747635f286166264929b3cc97529f2acb0e54686d1e` | 1 |
| 43 | `sha256:91f183a8f9674a4676dfe1a5274ea3c45160262a09d4169bd392af1b40b1a9bc` | 1 |
| 404 | `sha256:949c75c525a04164de19ad80ae06b92925b786756f20a7e95de17909dc7b3c12` | 1 |
| 72 | `sha256:95e75272f8d32a5b4779c49ce77bc249013d89d341325088f9edd0eea43ee003` | 3 |
| 251 | `sha256:9624d687dda71380aa1e890849cc2254fd557cb4c665e5565b9e93e227ff6f16` | 1 |
| 331 | `sha256:967c02cfb402a4fcc38dee5dd22c388cc6bdf5a1e1ef6c6e09299794be3736ac` | 2 |
| 445 | `sha256:96dde732d683e24c48af26da5fc6fc7281da6f8582be842ae1f6b3fca0981eba` | 1 |
| 530 | `sha256:978d59a9bc372e26b559f000c5bac22db3f84c5538de8fe7569b5030b2896339` | 4 |
| 95 | `sha256:99d0274f6dd5f6af9cef72f4689aa2da94f2d1b7129f23431b18ebba9ffac6c5` | 1 |
| 409 | `sha256:9ad5f6a4100c65f8f49be81b0f2a0adf2b3271c7342ce6d6d1c41453db3906fd` | 2 |
| 436 | `sha256:9b94585e7b43f11d71e933650825fc10efd5ff7c36f9958e4ae2b1652d963571` | 2 |
| 316 | `sha256:9d334424ba4b20a42b14e87105569924a3a6b360e59ddcf6963b72f7f680c548` | 4 |
| 486 | `sha256:9f30de08a64b76794349bc701bb0e246ca526e5b7dd5850fc7e350c904bb2036` | 1 |
| 123 | `sha256:9f9eecc05fc6be5d32bdfdea69830e0f4bd63333447c157a463d5df21baaff5c` | 2 |
| 167 | `sha256:9fe9bcd976abbdbb52ce185c532fe6c2cbf6df2176bb45dc917aea1bce0837fe` | 2 |
| 280 | `sha256:9ffef6d6bcb8df5207123f86ac8f7e4637bef6a999f420d0b9fb5a8b7d4656aa` | 1 |
| 502 | `sha256:a0082cadf6e8fa08553b5daeacd7d91e219126147a9207749be36123e202d49d` | 1 |
| 346 | `sha256:a142b9dd481f9c13115b38127cb99c8628df20e040b8ab926d381d98f9045961` | 1 |
| 207 | `sha256:a2169a230af3e4e6578571843253cc143387a2538877247e6f9731a2ce44cd66` | 2 |
| 416 | `sha256:a23aa3131b40cf701ac99e31542ef5e88e7643a5bb8c40f5a9f785406b17ab2a` | 3 |
| 225 | `sha256:a67f31b9e7b1046e53fde8c82718250808fd8f6909764e934b1f2b1d94f3e86a` | 2 |
| 75 | `sha256:a76971a4efa4004180fbb2f678ed7c33c83d21aeaef9a53deee4c4490cf00bbb` | 1 |
| 57 | `sha256:a8cd1908496e24c76f61fb369abf45ead6d868ff7a86f122abdbd202736c307c` | 1 |
| 371 | `sha256:aa227666aa903798b03de5a71a9a306a63692b37bee5a538101d35bb8e91cdd8` | 1 |
| 281 | `sha256:ac388377b5ceed69273a18807b6c7de4cedc62908653bdf6097200c2391c1487` | 2 |
| 447 | `sha256:b02d26e3a254fa3cfd003430240f8ea697e5ce7ce10f8c68b83738488f8f7d9a` | 3 |
| 227 | `sha256:b35df05b78e55ec1f541751da611e98a34e565be18a5fd27dde438aec7ef5ddd` | 1 |
| 407 | `sha256:b37c49b4a97ca2e7fe71b2b549c0b97c6ca21490025e1d9241c64880eb64fb12` | 1 |
| 342 | `sha256:b6a09ef353c02b7e45e76484f0b6849c250795d2b2922593e01cb73c1baf75e8` | 1 |
| 244 | `sha256:b738881462a4b26dc08b806e1824bbfbf9f54624a27f19cf648988468c56f529` | 1 |
| 245 | `sha256:b7450ae2d8a9a4a2b0572ee63b30ea58928fc814bd01e3e2256b84e44c8db2ef` | 2 |
| 139 | `sha256:b75658f6a3ca8106ab3f5692e5a2ae63b947a0c678fe3f81ae7b850f98c47e92` | 3 |
| 212 | `sha256:b76e6a2af0f4ed7a5f904e58c382da5b728d92dc3ae4d51ca2a412a7c3eed346` | 2 |
| 438 | `sha256:b76f53524ea94ed258744c73ea97409c567a09bb3ee9cbe909fa421005fe9189` | 2 |
| 48 | `sha256:b8699ec558b5b16f6b8d6ea958c4a7fa71449e1dcd45672ced7dc256736f4bdb` | 5 |
| 49 | `sha256:b8cdaf62198f4cac71fb33199728921912c86c092f73f02c7c6316c7a045b751` | 2 |
| 178 | `sha256:baa2fde0f741480d14104412f72f073cb6df6e58bbb17db6a09c2f8dccb02317` | 1 |
| 104 | `sha256:bbc2b408020f62bf58323ae9b4411a542c2d5c2125c19e9d7bf3c01bc46b9bb9` | 1 |
| 421 | `sha256:bca3d405478bb6fc26946e35c53d26cc99fdc46b8d1143c509d7b569833cb3ef` | 4 |
| 267 | `sha256:bd3f8288af958279d50cbe92c09717d26e6bf1aabaaf4811577a8ba64dadc895` | 1 |
| 474 | `sha256:bd64c014649657e2611b9b48f04037538738df42a43283cbc78f2479c0e49e23` | 2 |
| 430 | `sha256:bd8cefdeb0ae1f22a37ecf6751f91bb35eed9781d89411962080cbc856a6cd5c` | 1 |
| 5 | `sha256:be891fd6cb07ef466d5dbc766b28bfc7f6672349ab6311e64a5297e3419bfa32` | 1 |
| 326 | `sha256:bfd34e6444a475b153f16300145c9da4925efc565b7b7573b1c150fe4a2036bc` | 4 |
| 91 | `sha256:bfe1107d6da6dc9e897cc25d44f99ec347e1aae151e86b9dd9ccc02c595cebd4` | 1 |
| 498 | `sha256:c0203796197ca61454446ac9dea4668677db3c1277eb33810e97d0ffa1235bd5` | 3 |
| 283 | `sha256:c0f9cd0d6830054d406d7e0c7c6d28fb1d58132dd62c82f0cd07a5e8af5cdf25` | 2 |
| 30 | `sha256:c12120f1262049ede0fb66726354088ecb091382062b6be183293b8491c93666` | 1 |
| 14 | `sha256:c5759f0daa926b74d732bf0ed72ae47a3fee230c761d7d2ba5a2f67f015f704f` | 1 |
| 113 | `sha256:c59dac7b9787d505928dda6986cfe4354eb975bc7757741f543ff3940097cb11` | 5 |
| 504 | `sha256:c60c53eda985455e0102dc509eff188e1090874eca4efa5a25084abcfff659eb` | 1 |
| 476 | `sha256:c612d74f181508b571bc6cbe42017821dca65e7cfea0c51a6d5ac2bf1a047acb` | 3 |
| 199 | `sha256:c8b030115ef6430f92079f83d13ad6488988d8ab78b32ca8cc487d0f71009a2c` | 1 |
| 457 | `sha256:c9fbe4c984b47fb2f2ab5553723dcfd95f14ea6db8461a1ca610d9c1e05c6efd` | 1 |
| 237 | `sha256:cb5aa59f1fc2bfff0b3c10fe4f6fb4014081875859b53cefc75d227fa7b99855` | 1 |
| 496 | `sha256:ced73d73d3dbcb8978b25bc05f4c1f3295ea4aa35599077e55d6b4de9517c496` | 1 |
| 105 | `sha256:d16ed34b880d68e3cf2fc763f3874961b915f751f7c90db47d872fbd22514d51` | 2 |
| 422 | `sha256:d2fd3a25e784e2dd37e1707d3c278c2ee392981b49dec1a0c14f63b662be7fd4` | 3 |
| 168 | `sha256:d39e77243038b0921644e930cc401606eb89619ff2e48ce3c954404c793fb6de` | 2 |
| 315 | `sha256:d47081b4b9f7eefabae9716a306c547fe3973e0c95f878c0c199042499cb6286` | 3 |
| 184 | `sha256:d7d6ef92d5b6e807f2ec5f38879e2449e5cb8e54d39ed5d2bf8681ad4ce83bde` | 1 |
| 272 | `sha256:d7e9fc2439505565f3933b124a649453193f06dc67c92873df3cd7dc8bc5ffc7` | 1 |
| 456 | `sha256:d9487a9b5acfac7e5d28d9ab0b12fab384dfc15570f3a0c81d050828c5e53fe3` | 2 |
| 328 | `sha256:da1977a6cb590f6d4c45aead9e4acf5f5909914d30980452d1a76e415e83f99b` | 2 |
| 120 | `sha256:dc6803bf76843ae4d5f017264e81bd0f13666d3d69b28ae4ba2aaad06e340c21` | 2 |
| 46 | `sha256:dcce24ad1ff25e57ba8e0fab7c3f588bbfd7278c5d435a3f85fe141279e7f114` | 1 |
| 144 | `sha256:dd95e76ba615e1677a7d5129ead9dd84374928e2fe77535661887f2238acb039` | 1 |
| 54 | `sha256:dedc554b2fb6a31ffdab4c3c4b5ab7b6677304330fe90b3e11a0f1e7e55f28a7` | 3 |
| 37 | `sha256:e02cc4c89aa02e82a4c6d3ce8a8ab2956cc7fdb56805a8a262faba8dc50c0592` | 2 |
| 329 | `sha256:e19c0f8d3802f7d6cba6d809d8285d855da8004f0aaf281e0d3d169007c20acf` | 1 |
| 50 | `sha256:e1e11095ecb3bbf10ae7419d54c0b0c589dc94539f71121b81d7bba357efe59f` | 4 |
| 392 | `sha256:e22f8a51aae7a22ae59f444254d396448e73936aa81521cd491506a781ccba0c` | 1 |
| 153 | `sha256:e2791b50bd06ff79f9405427e8e1d5315a99a2a63c86da4dc51bd0f3c32eeb01` | 1 |
| 276 | `sha256:e3d5be116ebbafcbe103c36931460f4b7c9c4964d3a5317d260f0e746f362ffe` | 1 |
| 442 | `sha256:e4f0e39fb319bb56188f9c1197caea8ede34324274fd8df11a6a481a442fda9d` | 1 |
| 111 | `sha256:e584b1c755df2de67d43c6220607a654469b26212e3c1e74ba097e92fb1cc99a` | 3 |
| 242 | `sha256:e5dc7c8a2e6928112c791d1d75d27aa9be4b1bc346c910a3f221fdd09c61e547` | 1 |
| 298 | `sha256:e6405e70eed1b4ad0b145a4d1d2602aaec2bd45fab24ce95772d3ff810db238a` | 1 |
| 351 | `sha256:e97766fa020e80d8d213df937feb2ffa9b3f7642f3d7fb4c2227a49fa7fea576` | 2 |
| 142 | `sha256:e986567c2a76023c925bce661d9c6fce06eb58536cac131b526fa0b3e36bfa43` | 2 |
| 454 | `sha256:e9d8ce042edaf142966d91b090b2ce01640ded7aca15f973f69407e7fa50cc67` | 2 |
| 402 | `sha256:eaa696f6a81093443cf7596382692f1b05cccdf9e87c482ce60d44e668b48cde` | 1 |
| 132 | `sha256:eb132747612e734438c6740ebeed91750e3bc4f42a78eb418a1dfcfe6a42b103` | 1 |
| 414 | `sha256:ed40844ebaf25f7ed1911ea9834f31dea289d719c730c48e58cb0cc752677391` | 1 |
| 319 | `sha256:ef2540082aeef4b6baaca2775e90b66f328b95c80c118e6b9d451f80d9e50d1d` | 3 |
| 455 | `sha256:ef706ee9fb1429dbcce5d23868dcb60ca87aa6f9df894f55c3a172f24ee46634` | 2 |
| 193 | `sha256:f0e0cc3df54149c828739d7e32c0fb56529898fcc952c8a85d308f8d996861dc` | 2 |
| 187 | `sha256:f123fece1a667f6c0bfd27c6881d6f8d984fb9e3bfdc8615d95813a75b460b3f` | 1 |
| 204 | `sha256:f1ea95ad4b982cc8219695736a735d8500c9464ae4f07c1e2d040147fa31b86f` | 3 |
| 82 | `sha256:f37395f38f6753dad1bdd05504d2c9b7b1b0b82ae87bd46fe5476b43b0a298ae` | 2 |
| 284 | `sha256:f3c089bac4f6fd76f68303c905eddf2b55e3685fb9bda1ec1a523b4d166ad3ae` | 2 |
| 223 | `sha256:f4e69a7e520f0b410aa75315ca146d404338001711e58e72601ea8d4f0e2a0c3` | 1 |
| 538 | `sha256:f6109393865e513b000f56ede2890831851644415c123dc6727c738d5000ddb1` | 2 |
| 133 | `sha256:f661481352e7933c17125ca922f5be796a9d964aa8591f5cb7d917af82423cda` | 1 |
| 524 | `sha256:f67f27167e9389a9dd487a55d0649a29ba82786de2cb1b916c5b2036ffe2bb96` | 1 |
| 221 | `sha256:f687c7453ba4e55011dae4e3f8fe7e414c80b6c90feca99b2f57ebe7d206864f` | 2 |
| 107 | `sha256:f6c0d2bbd4c17271bd0bdcd1b91c80a62dc359c0942972f4f27670350ab3bea8` | 1 |
| 160 | `sha256:f6f017f0286ca8dc41a4f8327f7a4a4d60f07dd1d10f669db394f835a0804e36` | 1 |
| 100 | `sha256:f8b7fb3896a09ebc6799d528fd4b2a5d37537f5696862ef0f3d3d112e11e5f9c` | 4 |
| 154 | `sha256:f987c8e6810a421041f4d80f2ab08057486e7dfb2880182cc5d1ea8fe1e2ef21` | 1 |
| 183 | `sha256:fa354e0340acbaf45046aa9cb580c44a358e1aa4e084d93a2d29c32102c8b120` | 1 |
| 121 | `sha256:fa4870ff9b74298f596424f8b0d24b98c86adf4be95ce7e424506e259154ee8d` | 1 |
| 4 | `sha256:fb06c28c3b1e1c89c5def80e92f4b004762ffbdf9fa65cb1dd40038bc0def2c6` | 2 |
| 1 | `sha256:fb1168956ba24e9a436196195c7f764fd46b24d5c5aba28027a40c8651d68a01` | 1 |
| 423 | `sha256:fb9d8204755574fa87507e2947a0e6b611142c8ea062c77d047e10442c1f0ec2` | 1 |
| 180 | `sha256:fc90c771b0cf39525bf0adf3bacb35e4bf2e2cea0fbb8c75350642c497f0b529` | 2 |
| 140 | `sha256:fcc878238fed666496a1606a7d2c391bbf6e7fe254ea408e574f73553420b31b` | 4 |
| 428 | `sha256:fd5b714bfbd23f6656f613bf8a7b31ff1e2d07ab291b4272924697ae1abef28f` | 2 |
| 34 | `sha256:fd73bf7142ba6e032351df2f84074506cebf5df639a654eee55f275248ad126b` | 1 |
| 202 | `sha256:fd85ccca435960408cea23fa1403719cd71bda61abfccf2f45c0935c7afa8768` | 1 |


In [4]:
execute_local("adapter_geometry.py")
data = json.loads((package / "geometry.json").read_text())
print(json.dumps({k: v for k, v in data["audit"].items() if k != "checks"}, indent=2))


adapter_geometry.py: completed locally, exit 0
{
  "checkpoints": 34,
  "adapter_bytes_hashed": 12865858400,
  "modules_per_checkpoint": 249,
  "transformer_layers": 32,
  "unassigned_modules": [
    "base_model.model.model.unembed_tokens"
  ],
  "selected_tensor_spot_checks": 6
}


# Pair-2 adapter geometry — descriptive only

Local archived tensors and their hash-bound geometry caches only. No remote calls, new sampling, or gate input.

## Definitions and provenance

All **34** retained Stage-A checkpoints were checked against local segment and matching records; all 34 adapter payload hashes and all 34 cached-geometry hashes matched the archive manifest. Total adapter bytes hashed: **12,865,858,400**.

Each checkpoint contains **249 adapted modules** across transformer layers **0–31**, plus one unassigned output projection (`base_model.model.model.unembed_tokens`). B-S cadence checkpoints: u10–u290 every 10 updates; B-G: u10–u50 every 10 updates. The selected B-S u40 and B-G u20 are members of those 34 checkpoints. No u0/M0 or B-S u294 adapter is present in this archive.

All cached module spectra are recombined locally to validate every layer-level cache. For each selected checkpoint, the first, middle, and last module's factor norms and float64 thin-QR/SVD spectrum were independently recomputed from its tensor file (six modules total; one CPU thread). All checks passed. The other 8,460 cached module spectra were provenance-verified, not recomputed.

`||A||F` and `||B||F` are Frobenius norms; layer/model aggregation is the root-sum-of-squares across factors. `||BA||F`, singular spectra, and rank summaries use the **block-diagonal union of module BA matrices**. This is a bookkeeping aggregate, not the composition, Jacobian, or effective rank of the whole transformer layer/model. Full ordered spectra for every module and layer at every cadence point are in [geometry.json](geometry.json).

Entropy effective rank is `exp(-Σ p_i log p_i)`, with `p_i = σ_i/Σσ`; stable rank is `Σσ_i²/σ₁²`. Numerical rank counts singular values above `10⁻⁶ σ₁`. A/B norms are gauge-dependent: an invertible factor change can alter them without changing BA. These are full adapter values, not changes from M0. All 34 archived configs specify ordinary rank-32 LoRA with alpha 32 (alpha/r = 1), no rsLoRA/DoRA; the reported unscaled BA also has the configured scale of 1.

Source manifest: `artifacts/pilot0-pair2-lora-geometry/archive-manifest.json` (`sha256:9e36c2a9db1b915f88e9c042cc7013104c4d57d084c5ee1befda8b02b2086bee`). Individual source hashes and selected spot-check residuals are in the JSON audit.

## Selected checkpoint: per-layer factor and BA norms

| Layer | Modules | B-S u40 ‖A‖F | B-G u20 ‖A‖F | B-S ‖B‖F | B-G ‖B‖F | B-S ‖BA‖F | B-G ‖BA‖F |
|---|---:|---:|---:|---:|---:|---:|---:|
| 0 | 8 | 9.269591 | 9.231679 | 0.923280 | 0.202478 | 0.542178 | 0.117049 |
| 1 | 8 | 9.276870 | 9.241153 | 0.939625 | 0.203603 | 0.553451 | 0.117643 |
| 2 | 8 | 9.279359 | 9.241077 | 0.945507 | 0.204386 | 0.558526 | 0.118148 |
| 3 | 7 | 8.680786 | 8.642364 | 0.934501 | 0.200489 | 0.551290 | 0.115665 |
| 4 | 8 | 9.288568 | 9.233881 | 0.991789 | 0.206936 | 0.598848 | 0.119780 |
| 5 | 8 | 9.303322 | 9.242556 | 1.008804 | 0.208350 | 0.619099 | 0.121011 |
| 6 | 8 | 9.308088 | 9.241879 | 1.019862 | 0.208555 | 0.624296 | 0.121050 |
| 7 | 7 | 8.701400 | 8.645471 | 0.987822 | 0.204848 | 0.604174 | 0.118830 |
| 8 | 8 | 9.306088 | 9.243337 | 1.030789 | 0.210722 | 0.629235 | 0.122449 |
| 9 | 8 | 9.306893 | 9.242465 | 1.038341 | 0.210572 | 0.642766 | 0.122301 |
| 10 | 8 | 9.308576 | 9.237384 | 1.038855 | 0.210772 | 0.642603 | 0.122303 |
| 11 | 7 | 8.711527 | 8.644874 | 1.025015 | 0.208344 | 0.633350 | 0.120905 |
| 12 | 8 | 9.321617 | 9.241187 | 1.062471 | 0.214338 | 0.667250 | 0.124351 |
| 13 | 8 | 9.330265 | 9.244498 | 1.080299 | 0.213928 | 0.678706 | 0.124066 |
| 14 | 8 | 9.327276 | 9.237635 | 1.074789 | 0.214075 | 0.669130 | 0.123814 |
| 15 | 7 | 8.717574 | 8.641116 | 1.044630 | 0.209186 | 0.649513 | 0.121329 |
| 16 | 8 | 9.338314 | 9.246449 | 1.094747 | 0.211994 | 0.689761 | 0.122771 |
| 17 | 8 | 9.336472 | 9.239177 | 1.105108 | 0.210544 | 0.686658 | 0.122142 |
| 18 | 8 | 9.332710 | 9.241894 | 1.103379 | 0.210874 | 0.685444 | 0.122396 |
| 19 | 7 | 8.727404 | 8.646862 | 1.084292 | 0.205878 | 0.671241 | 0.119355 |
| 20 | 8 | 9.319708 | 9.234547 | 1.103803 | 0.212121 | 0.669761 | 0.122731 |
| 21 | 8 | 9.317592 | 9.229369 | 1.105588 | 0.212498 | 0.680548 | 0.122935 |
| 22 | 8 | 9.325026 | 9.242148 | 1.089565 | 0.212347 | 0.661930 | 0.122641 |
| 23 | 7 | 8.714943 | 8.644999 | 1.061495 | 0.207397 | 0.641446 | 0.120086 |
| 24 | 8 | 9.312993 | 9.235638 | 1.068517 | 0.214437 | 0.640996 | 0.123819 |
| 25 | 8 | 9.301338 | 9.227318 | 1.062964 | 0.214108 | 0.639022 | 0.123673 |
| 26 | 8 | 9.315431 | 9.240891 | 1.064021 | 0.214257 | 0.642341 | 0.123761 |
| 27 | 7 | 8.706471 | 8.641814 | 1.025823 | 0.209779 | 0.618703 | 0.121780 |
| 28 | 8 | 9.308979 | 9.238105 | 1.054933 | 0.214596 | 0.634881 | 0.124233 |
| 29 | 8 | 9.315429 | 9.243008 | 1.046425 | 0.214775 | 0.631624 | 0.124548 |
| 30 | 8 | 9.299336 | 9.237743 | 1.052306 | 0.213697 | 0.629834 | 0.123515 |
| 31 | 7 | 8.717919 | 8.641672 | 1.084484 | 0.211227 | 0.668769 | 0.122666 |
| unassigned | 1 | 3.287763 | 3.262564 | 4.251443 | 0.602143 | 2.650880 | 0.349917 |

## Selected checkpoint: per-layer spectrum and ranks

Full spectra, including small singular values, are retained in JSON; this table shows their largest value and two rank summaries.

| Layer | B-S σ1 | B-G σ1 | B-S entropy rank | B-G entropy rank | B-S stable rank | B-G stable rank |
|---|---:|---:|---:|---:|---:|---:|
| 0 | 0.138489 | 0.030776 | 199.365044 | 206.014773 | 15.326794 | 14.464578 |
| 1 | 0.131433 | 0.032773 | 189.705094 | 199.621878 | 17.731658 | 12.885361 |
| 2 | 0.139541 | 0.030552 | 184.143463 | 195.983127 | 16.020829 | 14.954259 |
| 3 | 0.199158 | 0.039314 | 152.011130 | 164.076519 | 7.662363 | 8.655912 |
| 4 | 0.210177 | 0.035464 | 181.390862 | 199.427155 | 8.118236 | 11.407318 |
| 5 | 0.251123 | 0.039966 | 173.939016 | 197.025790 | 6.077793 | 9.167729 |
| 6 | 0.263451 | 0.043887 | 173.803438 | 194.567467 | 5.615402 | 7.607858 |
| 7 | 0.241932 | 0.045467 | 139.119056 | 159.578421 | 6.236450 | 6.830579 |
| 8 | 0.252170 | 0.041817 | 172.304196 | 194.356355 | 6.226443 | 8.574210 |
| 9 | 0.277620 | 0.042886 | 162.984484 | 191.599111 | 5.360471 | 8.132643 |
| 10 | 0.276761 | 0.045660 | 168.132636 | 193.923891 | 5.391072 | 7.174845 |
| 11 | 0.260663 | 0.039940 | 136.931250 | 158.694368 | 5.903772 | 9.163652 |
| 12 | 0.289209 | 0.045453 | 167.443976 | 192.456927 | 5.322957 | 7.484634 |
| 13 | 0.279399 | 0.043903 | 171.891150 | 196.376964 | 5.900846 | 7.985799 |
| 14 | 0.275089 | 0.044039 | 174.193321 | 195.736312 | 5.916642 | 7.904351 |
| 15 | 0.271233 | 0.039354 | 145.815433 | 161.073673 | 5.734433 | 9.505112 |
| 16 | 0.283879 | 0.040192 | 179.229402 | 199.164166 | 5.903789 | 9.330684 |
| 17 | 0.267039 | 0.038324 | 185.116360 | 203.051739 | 6.611975 | 10.157384 |
| 18 | 0.274073 | 0.041496 | 186.903195 | 198.656218 | 6.254766 | 8.700029 |
| 19 | 0.255617 | 0.038522 | 152.824275 | 164.268892 | 6.895699 | 9.599864 |
| 20 | 0.217650 | 0.038658 | 190.144087 | 200.090601 | 9.469380 | 10.079119 |
| 21 | 0.248773 | 0.035230 | 189.801201 | 203.645197 | 7.483644 | 12.176583 |
| 22 | 0.228718 | 0.037721 | 193.533458 | 204.586264 | 8.375750 | 10.570863 |
| 23 | 0.235416 | 0.036120 | 157.048881 | 165.122403 | 7.424164 | 11.052989 |
| 24 | 0.197064 | 0.042884 | 194.926162 | 200.019351 | 10.580245 | 8.336434 |
| 25 | 0.203167 | 0.041466 | 195.705391 | 202.202903 | 9.892953 | 8.895240 |
| 26 | 0.231499 | 0.035714 | 196.440734 | 202.625384 | 7.698991 | 12.008144 |
| 27 | 0.221121 | 0.044774 | 157.131915 | 159.248681 | 7.828957 | 7.397852 |
| 28 | 0.215007 | 0.039014 | 195.401971 | 200.520935 | 8.719281 | 10.140061 |
| 29 | 0.220370 | 0.042327 | 192.662585 | 198.058969 | 8.215060 | 8.658287 |
| 30 | 0.208348 | 0.045248 | 192.216693 | 190.805116 | 9.138429 | 7.451279 |
| 31 | 0.266609 | 0.047330 | 147.514271 | 143.823586 | 6.292193 | 6.717049 |
| unassigned | 2.463769 | 0.297040 | 13.692713 | 17.581577 | 1.157657 | 1.387712 |

## Cadence trajectories: all-module block-diagonal aggregate

The JSON also gives the full per-layer cadence trajectories and spectra. No weighting or normalization for different update counts is applied.

| Arm | Update | Selected | ‖A‖F | ‖B‖F | ‖BA‖F | σ1 | Entropy rank | Stable rank |
|---|---:|---|---:|---:|---:|---:|---:|---:|
| B-G | 10 |  | 51.543173 | 1.278235 | 0.740562 | 0.269059 | 5828.625009 | 7.575793 |
| B-G | 20 | yes | 51.545578 | 1.332343 | 0.772266 | 0.297040 | 5888.770501 | 6.759322 |
| B-G | 30 |  | 51.550705 | 1.390787 | 0.806562 | 0.321348 | 5916.234346 | 6.299753 |
| B-G | 40 |  | 51.557592 | 1.448030 | 0.840179 | 0.335651 | 5928.229607 | 6.265677 |
| B-G | 50 |  | 51.565608 | 1.495617 | 0.868122 | 0.328037 | 5931.656053 | 7.003525 |
| B-S | 10 |  | 51.698052 | 4.187164 | 2.479261 | 1.221943 | 5697.169299 | 4.116636 |
| B-S | 20 |  | 51.817118 | 5.783587 | 3.493723 | 1.838998 | 5517.261369 | 3.609230 |
| B-S | 30 |  | 51.894048 | 6.688752 | 4.087714 | 2.216093 | 5415.052709 | 3.402396 |
| B-S | 40 | yes | 51.943300 | 7.274779 | 4.475286 | 2.463769 | 5353.155592 | 3.299449 |
| B-S | 50 |  | 51.983599 | 7.729291 | 4.782424 | 2.652603 | 5302.870469 | 3.250510 |
| B-S | 60 |  | 52.011703 | 8.144708 | 5.062197 | 2.857384 | 5263.759258 | 3.138637 |
| B-S | 70 |  | 52.044031 | 8.511924 | 5.315410 | 3.034050 | 5234.209587 | 3.069220 |
| B-S | 80 |  | 52.076251 | 8.856684 | 5.558515 | 3.205537 | 5203.450363 | 3.006880 |
| B-S | 90 |  | 52.103013 | 9.238682 | 5.816574 | 3.420284 | 5174.812345 | 2.892079 |
| B-S | 100 |  | 52.117787 | 9.664029 | 6.088569 | 3.678292 | 5148.567645 | 2.739921 |
| B-S | 110 |  | 52.140999 | 10.142421 | 6.397438 | 3.960619 | 5112.336940 | 2.609072 |
| B-S | 120 |  | 52.162458 | 10.689853 | 6.748734 | 4.309573 | 5073.641721 | 2.452315 |
| B-S | 130 |  | 52.191870 | 11.253482 | 7.119936 | 4.669970 | 5031.795786 | 2.324470 |
| B-S | 140 |  | 52.211995 | 11.757012 | 7.439203 | 4.961068 | 5002.733095 | 2.248550 |
| B-S | 150 |  | 52.237077 | 12.231630 | 7.760616 | 5.230293 | 4972.051856 | 2.201610 |
| B-S | 160 |  | 52.258878 | 12.659956 | 8.049257 | 5.454120 | 4945.974817 | 2.178022 |
| B-S | 170 |  | 52.283691 | 13.287849 | 8.481738 | 5.860135 | 4903.319837 | 2.094857 |
| B-S | 180 |  | 52.320497 | 14.037247 | 9.007792 | 6.356263 | 4853.470104 | 2.008321 |
| B-S | 190 |  | 52.355631 | 14.558374 | 9.360850 | 6.577334 | 4823.422256 | 2.025493 |
| B-S | 200 |  | 52.381849 | 14.879828 | 9.554414 | 6.573574 | 4806.193991 | 2.112540 |
| B-S | 210 |  | 52.405859 | 15.341966 | 9.819974 | 6.678881 | 4788.510866 | 2.161789 |
| B-S | 220 |  | 52.434116 | 16.000090 | 10.252268 | 7.001359 | 4754.822884 | 2.144248 |
| B-S | 230 |  | 52.463329 | 16.614915 | 10.673810 | 7.332384 | 4720.490852 | 2.119086 |
| B-S | 240 |  | 52.500490 | 17.219897 | 11.102230 | 7.648892 | 4684.271848 | 2.106800 |
| B-S | 250 |  | 52.545946 | 17.803837 | 11.511574 | 7.875954 | 4650.504488 | 2.136304 |
| B-S | 260 |  | 52.574012 | 18.231061 | 11.761604 | 7.951977 | 4637.435059 | 2.187675 |
| B-S | 270 |  | 52.604754 | 18.624069 | 12.009831 | 8.045287 | 4624.103447 | 2.228388 |
| B-S | 280 |  | 52.636289 | 18.972861 | 12.262837 | 8.152663 | 4606.935595 | 2.262471 |
| B-S | 290 |  | 52.662135 | 19.410118 | 12.564764 | 8.306109 | 4584.072480 | 2.288302 |


# Confirmatory design notes — non-binding

_2026-08-30. Post-hoc notes from Pilot pair 1, seed 11._

**No design authority is exercised here.** These are candidates and questions,
not decisions, amendments, preregistration, launch authorization, or new gates.
Nothing in this document changes the frozen protocol, charter, matching rule,
measurement schedule, or running pair-2 job. The analysis used pair-1 local
artifacts only. Any future design would require a separate prospective decision
before its confirmatory outcomes are observed.

Source: [offline analysis package](../artifacts/pilot0-pair1-offline-analysis/README.md),
with paired item-bootstrap intervals, early-window statistics, recorded failure
codes, and archived adapter geometry. Intervals are conditional on one fitted
seed pair and its selected checkpoints; they are not seed-to-seed uncertainty.

## Early-window retention estimand candidates

Candidate summaries include absolute targeted Pass@1 AUC over a prospectively
fixed early window, own-baseline-relative AUC over that window, and time to a
fixed fraction of initial retention. The package reports 0–20-update AUC and
interpolated half-life because they were requested after pair 1 completed;
those windows and summaries are therefore exploratory, not confirmatory.

For context, targeted raw-rate half-lives in pair 1 are 2.664286 updates for
B-S and 4.104651 for B-G, interpolated within the observed [2,5] bracket.
Relative raw-rate AUC(0–20) is 0.183079 and 0.240612, respectively. An eventual
estimand would need its window, interpolation, non-crossing rule, denominator,
and seed-level aggregation fixed in advance. It should distinguish retention
of total pre-B performance from survival of specifically newly acquired ability.

Absolute and relative summaries answer different questions when starting
scores differ. Relative scores become unstable near zero. The four-draw
Jeffreys floor also matters: B-S sentinel posterior half-life cannot cross half
its initial score because that threshold lies below 0.1. Raw rates and posterior
scores must remain explicitly labelled, with any future floor treatment fixed
prospectively rather than chosen from these outcomes.

## Equivalence-margin matching on a high-draw statistic

A candidate future rule could require a confidence interval for a paired,
high-draw targeted capability difference to lie inside a scientifically chosen
equivalence margin. The margin would need external justification before new
outcomes; overlapping confidence intervals or a non-significant difference are
not an equivalence test.

Pair 1 matched at 31/96 for both checkpoints on the cadence panel. On the
separate 16-draw targeted validation panel, B-S minus B-G Pass@1 is 0.026367,
with an item-bootstrap 95% CI of [-0.003906, 0.057129]. This is a conditional
description of residual mismatch, not evidence for any particular margin.
Any future matching plan would need independent confirmation or explicit
selection-aware uncertainty, candidate-search and tie rules, draw budgets,
and a no-match outcome frozen in advance. These notes choose none of them.

## F2 estimand sign-dependence

For pair 1, B-S minus B-G update-480 Pass@1 is -0.025513; the corresponding
posterior-score difference is -0.024012. The stored posterior raw-gain AUC
difference is +0.056936, and the absolute posterior AUC difference is +0.009947.
Endpoint, integrated absolute performance, and improvement from an arm's own
baseline are distinct estimands; their signs need not agree.

A future confirmatory plan should choose its primary question and estimand
before observing its outcomes, report the other frozen summaries alongside it,
and avoid choosing a preferred headline from the observed signs. Baseline
concentration can be reported using a predeclared item-set definition. Selecting
items by observed baseline success also induces selection/regression-to-mean
effects; same-item trajectories alone do not remove that issue.

## Replay-experiment motivation

A candidate supervised-replay comparison could use a prospectively fixed,
verifier-correct subset of B-G rollouts as SFT data from the same origin. This
could help separate consequences of the training objective from consequences
of the trajectory distribution (length, strategy use, content, and formatting).
Replay is not an intervention on just one factor unless the data-selection,
exposure, token-budget, optimizer, and checkpoint-selection rules are specified.
No replay run, recipe, selection rule, or spend is authorized here.

The F2 failure-code check does not support cap/tag repair as a description of
the B-G 40–80 transition in this run: those counts are zero at both endpoints;
successes rise from 592 to 1,407 out of 8,192 while wrong-target counts change
from 7,600 to 6,785. This does not identify a causal learning mechanism.
Descriptive LoRA geometry likewise cannot establish that an effective-rank or
norm difference causes retention or transfer. Factor norms depend on LoRA
parameterization; blockwise BA spectra describe the saved adapters, not a
network-level causal quantity.

## prospective mechanical prediction recorded before pair-2 outcome inspection.

_Added 2026-08-30 at 21:11:34 UTC, before reading any pair-2 outcome; non-binding._

prospective mechanical prediction recorded before pair-2 outcome inspection.
Inherited LoRA factor
scale may act as an effective step-size multiplier under the fresh
Stage-B Adam optimizer (function-space movement via the A-pathway
scales with ‖B‖, ~7x larger for B-S@140 than B-G@30). If so, it
predicts jointly: faster F1 decay AND faster early F2 gain for the
larger-scale arm, in any pair. It predicts nothing about pre-B
transfer, strategy diversity, generalization gap, or F2 endpoint.
A merge-to-base + identically-initialized fresh Stage-B adapter
comparison would test it. Descriptive observation for the record:
both arms' dominant adapter singular direction lies in the unembed
module; B-S's is near-rank-1 with ~11x B-G's magnitude. No design
authority exercised.

## Candidate gauge-rescaling follow-up — non-binding

_Added 2026-08-30 at 21:34:34 UTC; no authorization._

A candidate follow-up would compare functionally identical LoRA
parameterizations `(A, B)`, `(2A, B/2)`, and `(A/2, 2B)`, plus a version
whose B norm is matched to the other arm with compensating inverse scaling
of A so that BA, and hence the starting function, remains fixed. Each version
would receive identical Stage-B data and the same recipe, with a fresh optimizer
and the same seed. The question is whether F1 and F2 vary with parameterization
at an exactly fixed starting function.

Adam epsilon and any weight decay break exact gauge invariance and are part
of the mechanism being tested. A refinement would distinguish unembed-only
equalization from equalization across all adapted modules. The B-scale asymmetry
may partly be driven by the 10× Stage-A learning-rate difference between the
two frozen procedures; the gauge test does not depend on the cause of that
asymmetry.

This is a non-binding candidate only: no design or launch authority is exercised,
no experiment or spend is authorized, and no running job, gate, measurement,
or computation is changed.

## Prospective Pair-2 geometry-prediction scoring rule

_Added 2026-09-03 at 09:22:40 UTC, after the geometry-first prediction was
recorded and before opening any Pair-2 F1/F2/F3 outcome._

Pair 2: seed 29, selected checkpoints B-S@40 and B-G@20, with ‖B‖F ratio
5.46.

The geometry-based prediction is:

1. Faster forgetting under B-S: B-S has the shorter targeted raw-Pass@1
   retention half-life, defined as the first downward crossing of 50% of that
   arm's own update-0 Pass@1, linearly interpolated between registered
   checkpoints. If one arm never crosses, it is treated as longer than an arm
   that does cross. If neither crosses, this leg is inconclusive.
2. Faster early Stage-B learning under B-S: B-S has the larger
   baseline-relative raw-Pass@1 gain AUC over the fixed grid
   `[0, 1, 2, 5, 10, 20, 40]`, trapezoidally integrated and divided by 40,
   where `gain(t) = Pass@1(t) − Pass@1(0)`.

Scoring:

- both predictions hold = CONFIRMED
- exactly one holds = PARTIAL MISS
- neither holds = FAILED
- an inconclusive F1 leg does not count as holding

Report but do not use for scoring:

- targeted relative retention at update 2;
- raw Pass@1 gain at update 40;
- stored 0–480 raw-gain AUC;
- F2 endpoint;
- F3 profiles;
- geometry and other diagnostics.

Also report the descriptive observation that Pair 2's ‖B‖F ratio, 5.46×, is
smaller than Pair 1's 7.36×, but do not score or interpret a
magnitude/dose-response relationship from only two pairs.

Source: [notes file](../../docs/confirmatory-design-notes.md).


In [5]:
# Full cadence/per-layer spectra remain in the sidecar, not duplicated into this notebook.
geometry = json.loads((package / "geometry.json").read_text())
selected = {r["method"]: r for r in geometry["checkpoints"] if r["selected"]}
layer_id = "0"  # Change to any layer "0".."31" or "unassigned" for local inspection.
for method in ("B-S", "B-G"):
    layer = next(r for r in selected[method]["layers"] if r["layer"] == layer_id)
    print(method, "selected update", selected[method]["step"], "layer", layer_id)
    print(json.dumps({k: v for k, v in layer.items() if k != "block_diagonal_ba_singular_values"}, indent=2))
print("Complete ordered spectra: geometry['checkpoints'][...]['layers'][...]['block_diagonal_ba_singular_values']")
print("Exact baseline and overlap item IDs: f2_diagnostics.json / baseline_concentration")


B-S selected update 40 layer 0
{
  "layer": "0",
  "module_count": 8,
  "a_frobenius_norm": 9.269590950149276,
  "b_frobenius_norm": 0.9232800447145983,
  "block_diagonal_ba_frobenius_norm": 0.5421779000719501,
  "largest_singular_value": 0.1384892769659158,
  "smallest_retained_singular_value": 0.0016342855185325445,
  "entropy_effective_rank": 199.36504390398198,
  "stable_rank": 15.326794220761581,
  "numerical_rank_relative_1e-6": 256
}
B-G selected update 20 layer 0
{
  "layer": "0",
  "module_count": 8,
  "a_frobenius_norm": 9.231679353798926,
  "b_frobenius_norm": 0.2024784792397473,
  "block_diagonal_ba_frobenius_norm": 0.11704883786480245,
  "largest_singular_value": 0.030776144953737514,
  "smallest_retained_singular_value": 0.0008299268170406779,
  "entropy_effective_rank": 206.01477278555933,
  "stable_rank": 14.464578228096608,
  "numerical_rank_relative_1e-6": 256
}
Complete ordered spectra: geometry['checkpoints'][...]['layers'][...]['block_diagonal_ba_singular_values']
